In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
df

In [ ]:
df['answer'].value_counts()

In [ ]:
import string

# Convert to lowercase and remove punctuation
cleaned = (
    df["prompt"]
    .astype(str)
    .str.lower()
    .str.translate(str.maketrans('', '', string.punctuation))
)

# Split by whitespace and collect all unique words
vocab = set()

for text in cleaned:
    vocab.update(text.split())

print("Vocabulary size:", len(vocab))

In [ ]:
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Get the prompt from Row ID 1
prompt = df.loc[df["id"] == 1, "prompt"].iloc[0]
# If ID is not a column and row 1 means first row:
# prompt = train_df["prompt"].iloc[0]

# Clean it
cleaned = prompt.lower().translate(
    str.maketrans('', '', string.punctuation)
)

# Remove stop words
words = cleaned.split()
filtered_words = [w for w in words if w not in ENGLISH_STOP_WORDS]

print("Words left:", len(filtered_words))
#print(filtered_words)  # optional

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Combine prompt and all options into one document per row
combined_text = (
    df["prompt"].fillna("") + " " +
    df["A"].fillna("") + " " +
    df["B"].fillna("") + " " +
    df["C"].fillna("") + " " +
    df["D"].fillna("") + " " +
    df["E"].fillna("")
)

vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(combined_text)

print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Feature columns:", X.shape[1])  # should be identical

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

row = df.loc[df["id"] == 1].iloc[0]

prompt_vec = vectorizer.transform([row["prompt"]])
option_a_vec = vectorizer.transform([row["A"]])

similarity = cosine_similarity(prompt_vec, option_a_vec)[0, 0]

print(round(similarity, 4))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Fit the TF-IDF vectorizer on all prompts and options
combined_text = (
    df["prompt"].fillna("") + " " +
    df["A"].fillna("") + " " +
    df["B"].fillna("") + " " +
    df["C"].fillna("") + " " +
    df["D"].fillna("") + " " +
    df["E"].fillna("")
)

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text)

correct = 0

for _, row in df.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    similarities = {}

    for option in ["A", "B", "C", "D", "E"]:
        option_vec = vectorizer.transform([row[option]])
        similarities[option] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0, 0]

    predicted_answer = max(similarities, key=similarities.get)

    if predicted_answer == row["answer"]:
        correct += 1

accuracy = (correct / len(df)) * 100

print(f"Accuracy: {accuracy:.2f}%")

In [ ]:
from collections import Counter

# Count answer frequencies
answer_counts = df["answer"].value_counts()

print(answer_counts)

# Top 3 most frequent answers
top3 = answer_counts.index[:3].tolist()

print("Top 3:", top3)

# MAP@3 for static prediction
score = 0

for true_answer in df["answer"]:
    if true_answer == top3[0]:
        score += 1.0
    elif true_answer == top3[1]:
        score += 0.5
    elif true_answer == top3[2]:
        score += 1/3

map3 = score / len(df)

print("MAP@3 =", round(map3, 6))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Fit vectorizer on all text
combined_text = (
    df["prompt"].fillna("") + " " +
    df["A"].fillna("") + " " +
    df["B"].fillna("") + " " +
    df["C"].fillna("") + " " +
    df["D"].fillna("") + " " +
    df["E"].fillna("")
)

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text)

map3_total = 0.0

for _, row in df.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    similarities = {}

    for option in ["A", "B", "C", "D", "E"]:
        option_vec = vectorizer.transform([row[option]])
        similarities[option] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0, 0]

    # Sort options by similarity descending
    ranked = sorted(
        similarities.items(),
        key=lambda x: x[1],
        reverse=True
    )

    top3 = [x[0] for x in ranked[:3]]

    # MAP@3 contribution
    if row["answer"] in top3:
        rank = top3.index(row["answer"]) + 1
        map3_total += 1 / rank

final_map3 = map3_total / len(df)

print("MAP@3 =", round(final_map3, 6))

In [ ]:
df.isna().sum()